# TicketIQ — Week 3 Data Engineering (Part A & B)
**Platform:** Google Colab | **Pathway:** B — CSV/Excel

This notebook performs source ingestion, inspection, two-subset sampling, preparation comparison, validation, export and lineage verification.

**Dataset limitation:** the Kaggle source is general customer support data, not verified authentic ITSM incident data. Treat it as a provisional proxy only. Do not claim generalisation to real IT help-desk tickets.

**Privacy:** do not publish raw/prepared CSV files or share this notebook publicly. Full ticket descriptions are not printed. Email patterns are redacted; phone-like patterns are flagged but not automatically redacted. Pattern checks do not guarantee anonymisation.

## Part A1 — Requirement-to-data mapping

| Requirement | Data needed | Source fields | Constraint |
|---|---|---|---|
| FR1 | Initial ticket text | `Ticket Subject`, `Ticket Description` | General support data, not verified ITSM. |
| FR2 | Text for TF-IDF | Same two text fields | Template-heavy content may affect signal; fit TF-IDF on training data only later. |
| FR3/FR4 | Same target for both models | `Ticket Type` | Five supplied labels; target quality is provisional. |
| RAI1 | Exclude unnecessary identifiers | Exclude `Customer Name`, `Customer Email`; redact detected emails | Phone-like patterns flagged; residual PII may remain. |
| RAI2 | Initial-submission information only | Subject and description; exclude resolution/status/response-time/satisfaction fields | No post-submission fields as model inputs. |
| RAI4 | Duplicate/conflict checks before splitting | Subject, description, target | Initial audit found 42 duplicate rows in selected projection and 85 conflicting input groups (205 records). |
| NFR3 | Repeatable preparation and traceability | Explicit source path, SHA-256 hashes, manifest | Model seed/configuration will be recorded in the later model pipeline. |

## Part A2 — Initialise Google Colab workspace
Run this cell first and again after any runtime reset.

In [30]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, re, shutil
import pandas as pd

BASE_DIR = Path("/content/week3_project")
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RAW_PATH = RAW_DIR / "customer_support_tickets.csv"
PREPARED_PATH = PROCESSED_DIR / "prepared_table.csv"
MANIFEST_PATH = BASE_DIR / "data" / "week3_manifest.json"

print("BASE:", BASE_DIR)
print("RAW:", RAW_DIR, RAW_DIR.is_dir())
print("PROCESSED:", PROCESSED_DIR, PROCESSED_DIR.is_dir())

BASE: /content/week3_project
RAW: /content/week3_project/data/raw True
PROCESSED: /content/week3_project/data/processed True


### Upload the source CSV
If the CSV is only in Codespaces, download `data/raw/customer_support_tickets.csv` from the Explorer to your computer first, then upload it here. Alternatively, download the same Kaggle file again. Keep this notebook private.

In [31]:
from google.colab import files
uploaded = files.upload()
for filename, content in uploaded.items():
    (RAW_DIR / filename).write_bytes(content)
    print("Uploaded:", filename, "| bytes:", len(content))
assert RAW_PATH.is_file(), "Upload customer_support_tickets.csv with this exact filename."
print("Source verified:", RAW_PATH)

Saving customer_support_tickets.csv to customer_support_tickets (1).csv
Uploaded: customer_support_tickets (1).csv | bytes: 3945533
Source verified: /content/week3_project/data/raw/customer_support_tickets.csv


### A2 — Verify the selected input

In [32]:
source_files = sorted(p for p in RAW_DIR.rglob("*") if p.is_file())
for p in source_files:
    print(p.relative_to(RAW_DIR), p.stat().st_size, "bytes")
assert RAW_PATH in source_files
print("A2 access verification: PASS")

customer_support_tickets (1).csv 3945533 bytes
customer_support_tickets.csv 3945533 bytes
A2 access verification: PASS


## Part A3 — Inspect source and two representative subsets
The notebook reports schema, data types, missing values, duplicates, target distribution, text quality and label consistency. To minimise privacy exposure, it shows subject/target/length metadata rather than full descriptions.

In [33]:
df = pd.read_csv(RAW_PATH, encoding="utf-8-sig")
required = ["Ticket ID","Ticket Type","Ticket Subject","Ticket Description","Ticket Priority"]
missing = [c for c in required if c not in df.columns]
if missing: raise ValueError(f"Missing required columns: {missing}")

print("SOURCE:", RAW_PATH.relative_to(BASE_DIR))
print("SHAPE:", df.shape)
print("COLUMNS:", df.columns.tolist())
print("\nDATA TYPES:\n", df.dtypes)
print("\nMISSING VALUES:\n", df.isna().sum())
print("\nFULL-ROW DUPLICATES:", int(df.duplicated().sum()))
print("DUPLICATE TICKET IDs:", int(df["Ticket ID"].duplicated().sum()))
print("\nTICKET TYPE DISTRIBUTION:\n", df["Ticket Type"].value_counts().sort_index())
for col in ["Ticket Subject","Ticket Description"]:
    s = df[col].fillna("").astype(str)
    print(col, "| empty", int(s.str.strip().eq("").sum()),
          "| unique", int(s.nunique()), "| mean length", round(s.str.len().mean(),2),
          "| placeholder rows", int(s.str.contains(r"\{[^{}]+\}",regex=True).sum()))

SOURCE: data/raw/customer_support_tickets.csv
SHAPE: (8469, 17)
COLUMNS: ['Ticket ID', 'Customer Name', 'Customer Email', 'Customer Age', 'Customer Gender', 'Product Purchased', 'Date of Purchase', 'Ticket Type', 'Ticket Subject', 'Ticket Description', 'Ticket Status', 'Resolution', 'Ticket Priority', 'Ticket Channel', 'First Response Time', 'Time to Resolution', 'Customer Satisfaction Rating']

DATA TYPES:
 Ticket ID                         int64
Customer Name                    object
Customer Email                   object
Customer Age                      int64
Customer Gender                  object
Product Purchased                object
Date of Purchase                 object
Ticket Type                      object
Ticket Subject                   object
Ticket Description               object
Ticket Status                    object
Resolution                       object
Ticket Priority                  object
Ticket Channel                   object
First Response Time         

In [34]:
EMAIL_RE = re.compile(r"\b[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,}\b", re.I)
PHONE_RE = re.compile(r"(?<!\w)(?:\+?\d[\d\s().-]{6,}\d)(?!\w)")
def phone_count(v):
    n = 0
    for m in PHONE_RE.finditer(str(v)):
        digits = re.sub(r"\D","",m.group())
        n += int(8 <= len(digits) <= 15)
    return n

for col in ["Ticket Subject","Ticket Description"]:
    print(col, "email patterns:", int(df[col].fillna("").astype(str).str.count(EMAIL_RE).sum()),
          "| phone-like patterns:", int(df[col].fillna("").astype(str).apply(phone_count).sum()))

print("\nA3 SUBSET 1 — FIRST FIVE SOURCE ROWS")
s1 = df.head(5)
display(pd.DataFrame({
    "source_row": s1.index+1,
    "subject": s1["Ticket Subject"].astype(str).str.slice(0,80),
    "target": s1["Ticket Type"],
    "subject_length": s1["Ticket Subject"].astype(str).str.len(),
    "description_length": s1["Ticket Description"].astype(str).str.len()
}))

print("\nA3 SUBSET 2 — ONE ROW PER TARGET CATEGORY")
s2 = df.groupby("Ticket Type",group_keys=False).head(1)
display(pd.DataFrame({
    "subject": s2["Ticket Subject"].astype(str).str.slice(0,80),
    "target": s2["Ticket Type"],
    "subject_length": s2["Ticket Subject"].astype(str).str.len(),
    "description_length": s2["Ticket Description"].astype(str).str.len()
}).reset_index(drop=True))
assert len(s1) >= 2 and s2["Ticket Type"].nunique() >= 2

Ticket Subject email patterns: 0 | phone-like patterns: 0
Ticket Description email patterns: 80 | phone-like patterns: 102

A3 SUBSET 1 — FIRST FIVE SOURCE ROWS


,source_row,subject,target,subject_length,description_length
0,1,Product setup,Technical issue,13,284
1,2,Peripheral compatibility,Technical issue,24,282
2,3,Network problem,Technical issue,15,275
3,4,Account access,Billing inquiry,14,262
4,5,Data loss,Billing inquiry,9,333



A3 SUBSET 2 — ONE ROW PER TARGET CATEGORY


,subject,target,subject_length,description_length
0,Product setup,Technical issue,13,284
1,Account access,Billing inquiry,14,262
2,Payment issue,Cancellation request,13,286
3,Refund request,Product inquiry,14,341
4,Battery life,Refund request,12,330


In [35]:
def norm(v):
    return re.sub(r"\s+"," ",str(v).strip().casefold())

audit = pd.DataFrame({
    "subject": df["Ticket Subject"].fillna("").apply(norm),
    "description": df["Ticket Description"].fillna("").apply(norm),
    "label": df["Ticket Type"].fillna("MISSING").astype(str)
})
dgroups = audit.groupby("description")["label"]
cgroups = audit.groupby(["subject","description"])["label"]
conflict_mask = cgroups.transform("nunique") > 1
print("Unique subjects:", audit["subject"].nunique())
print("Subject groups with multiple labels:", int((audit.groupby("subject")["label"].nunique()>1).sum()))
print("Description groups with multiple labels:", int((dgroups.nunique()>1).sum()))
print("Combined input groups:", int(len(cgroups)))
print("Combined groups with multiple labels:", int((cgroups.nunique()>1).sum()))
print("Records in combined conflicting groups:", int(conflict_mask.sum()))
print("A3 inspection completed: PASS")

Unique subjects: 16
Subject groups with multiple labels: 16
Description groups with multiple labels: 49
Combined input groups: 8325
Combined groups with multiple labels: 85
Records in combined conflicting groups: 205
A3 inspection completed: PASS


## Part A4 — Data inventory

| Source ID | Filename / URL | Type and purpose | Version / date / owner | Access / rights | Inspection evidence |
|---|---|---|---|---|---|
| S01 | `customer_support_tickets.csv` — [Kaggle](https://www.kaggle.com/datasets/suraj520/customer-support-ticket-dataset) | Structured CSV; candidate support-ticket text-classification source | Version 1; `suraj520`; 3.95 MB and approximately 3 years since update as displayed at inspection time | Public Kaggle download; CC0 1.0 Universal as displayed; recheck before redistribution | A3 schema, dimensions, dtypes, missingness, duplicates, target distribution, text audit and label consistency |
| S01-SUBSET-01 | First five records from S01 | Representative subset | Derived from S01 Version 1 | Same rights; keep private | First-five subset table in this notebook |
| S01-SUBSET-02 | One record per `Ticket Type` | Label-stratified subset | Derived from S01 Version 1 | Same rights; keep private | Category subset table in this notebook |

**Suitability limitation:** this is general customer support data, not confirmed ITSM data. Only one of five categories is `Technical issue`. Treat it as a proxy, not proof of real IT help-desk applicability.

## Part B1 — Select pathway
**Pathway B (CSV/Excel)** is the only applicable pathway. Inputs: `Ticket Subject`, `Ticket Description`. Provisional target: `Ticket Type`. Exclude all other columns, including identifiers, priority and post-submission/outcome fields. Raw source remains unchanged.

## Part B2 — Compare two preparation configurations

- **Approach A:** trim outer whitespace and redact detected email patterns; retain all rows.
- **Approach B:** same cleaning, exclude complete normalised subject-description groups with multiple labels, then remove exact duplicate modelling rows.

Approach B avoids identical inputs carrying contradictory targets and reduces duplicate leakage. It is selected only if all five labels remain. Exclusion does not prove which original label is correct; the raw source remains unchanged.

In [36]:
COLS = ["Ticket Subject","Ticket Description","Ticket Type"]
EXPECTED = {"Billing inquiry","Cancellation request","Product inquiry","Refund request","Technical issue"}

def clean_base(frame):
    x = frame[COLS].copy()

    if x.isna().any().any():
        raise ValueError("Missing selected values; no imputation applied.")

    for col in COLS:
        x[col] = x[col].astype(str).str.strip()

    if x.eq("").any().any():
        raise ValueError("Blank selected values after trimming.")

    for col in ["Ticket Subject", "Ticket Description"]:
        x[col] = x[col].apply(
            lambda v: EMAIL_RE.sub("[REDACTED_EMAIL]", v)
        )
        x[col] = x[col].apply(
            lambda v: PHONE_RE.sub("[REDACTED_PHONE]", v)
        )

    return x

approach_a = clean_base(df)

approach_a = clean_base(df)
work = approach_a.copy()
work["_s"] = work["Ticket Subject"].apply(norm)
work["_d"] = work["Ticket Description"].apply(norm)
mask = work.groupby(["_s","_d"])["Ticket Type"].transform("nunique") > 1
conflict_groups_removed = int(work.loc[mask,["_s","_d"]].drop_duplicates().shape[0])
conflict_records_removed = int(mask.sum())
unambiguous = work.loc[~mask,COLS].copy()
duplicates_removed = int(unambiguous.duplicated(COLS).sum())
approach_b = unambiguous.drop_duplicates(COLS,keep="first").reset_index(drop=True)

if set(approach_b["Ticket Type"].unique()) != EXPECTED:
    raise ValueError("Approach B lost one or more expected labels. Stop and review.")

comparison = pd.DataFrame([
 {"approach":"A - minimal cleaning","rows":len(approach_a),
  "missing":int(approach_a.isna().sum().sum()),
  "duplicate modelling rows":int(approach_a.duplicated(COLS).sum()),
  "classes":int(approach_a["Ticket Type"].nunique())},
 {"approach":"B - conflict/duplicate filtered","rows":len(approach_b),
  "missing":int(approach_b.isna().sum().sum()),
  "duplicate modelling rows":int(approach_b.duplicated(COLS).sum()),
  "classes":int(approach_b["Ticket Type"].nunique())}
])
display(comparison)
print("Conflicting groups excluded:",conflict_groups_removed)
print("Records in conflicting groups excluded:",conflict_records_removed)
print("Exact duplicate rows removed:",duplicates_removed)
print("\nApproach A labels:\n",approach_a["Ticket Type"].value_counts().sort_index())
print("\nApproach B labels:\n",approach_b["Ticket Type"].value_counts().sort_index())
selected = approach_b.copy()
print("Selected: Approach B")

,approach,rows,missing,duplicate modelling rows,classes
0,A - minimal cleaning,8469,0,42,5
1,B - conflict/duplicate filtered,8240,0,0,5


Conflicting groups excluded: 85
Records in conflicting groups excluded: 205
Exact duplicate rows removed: 24

Approach A labels:
 Ticket Type
Billing inquiry         1634
Cancellation request    1695
Product inquiry         1641
Refund request          1752
Technical issue         1747
Name: count, dtype: int64

Approach B labels:
 Ticket Type
Billing inquiry         1601
Cancellation request    1640
Product inquiry         1601
Refund request          1703
Technical issue         1695
Name: count, dtype: int64
Selected: Approach B


## Part B3 — Export and validate prepared table
The output filename `prepared_table.csv` follows the lab's Pathway B deliverable.

In [37]:
PROCESSED_DIR.mkdir(parents=True,exist_ok=True)
selected.to_csv(PREPARED_PATH,index=False,encoding="utf-8-sig")
exported = pd.read_csv(PREPARED_PATH,encoding="utf-8-sig")
temp = exported.assign(_s=exported["Ticket Subject"].apply(norm),
                       _d=exported["Ticket Description"].apply(norm))
out_conflicts = int((temp.groupby(["_s","_d"])["Ticket Type"].nunique()>1).sum())

checks = {
 "file_exists": PREPARED_PATH.is_file(),
 "file_non_empty": PREPARED_PATH.is_file() and PREPARED_PATH.stat().st_size>0,
 "schema": exported.columns.tolist()==COLS,
 "row_count": len(exported)==len(selected),
 "no_missing": int(exported.isna().sum().sum())==0,
 "all_five_labels": set(exported["Ticket Type"].unique())==EXPECTED,
 "no_exact_duplicates": int(exported.duplicated(COLS).sum())==0,
 "no_conflicting_input_groups": out_conflicts==0
}
for k,v in checks.items(): print(k, "PASS" if v else "FAIL")
if not all(checks.values()): raise ValueError(f"Validation failed: {checks}")
print("Source rows:",len(df),"| Approach A:",len(approach_a),
      "| Prepared rows:",len(exported),"| Removed:",len(df)-len(exported))
print("Prepared path:",PREPARED_PATH,"| bytes:",PREPARED_PATH.stat().st_size)
display(exported[["Ticket Subject","Ticket Type"]].head(5))
print("B3 validation: PASS")

file_exists PASS
file_non_empty PASS
schema PASS
row_count PASS
no_missing PASS
all_five_labels PASS
no_exact_duplicates PASS
no_conflicting_input_groups PASS
Source rows: 8469 | Approach A: 8469 | Prepared rows: 8240 | Removed: 229
Prepared path: /content/week3_project/data/processed/prepared_table.csv | bytes: 2717969


,Ticket Subject,Ticket Type
0,Product setup,Technical issue
1,Peripheral compatibility,Technical issue
2,Network problem,Technical issue
3,Account access,Billing inquiry
4,Data loss,Billing inquiry


B3 validation: PASS


## Part B4 — Register lineage and verify manifest

In [38]:
def sha256(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda:f.read(1024*1024),b""): h.update(chunk)
    return h.hexdigest()

record = {
 "source_id":"S01",
 "source_filename":RAW_PATH.name,
 "source_relative_path":str(RAW_PATH.relative_to(BASE_DIR)),
 "source_sha256":sha256(RAW_PATH),
 "prepared_output":str(PREPARED_PATH.relative_to(BASE_DIR)),
 "prepared_sha256":sha256(PREPARED_PATH),
 "processing_note":"Selected subject, description and Ticket Type; trimmed outer whitespace; redacted detected emails; excluded conflicting normalised input groups; removed exact duplicate modelling rows.",
 "source_version":"Version 1 (as displayed on Kaggle)",
 "source_owner":"suraj520",
 "access_rights":"CC0 1.0 Universal as displayed on Kaggle; recheck before redistribution",
 "status":"Prepared; residual privacy and dataset-suitability review required"
}
manifest = {
 "pipeline_version":"week3-v1",
 "generated_at_utc":datetime.now(timezone.utc).isoformat(),
 "project":"TicketIQ",
 "selected_pathway":"B - CSV/Excel",
 "records":[record],
 "comparison":{
  "approach_a_rows":int(len(approach_a)),
  "approach_b_rows":int(len(approach_b)),
  "conflicting_groups_excluded":conflict_groups_removed,
  "conflicting_records_excluded":conflict_records_removed,
  "exact_duplicate_rows_removed":duplicates_removed
 },
 "validation":checks,
 "limitations":[
  "General customer support source, not verified authentic ITSM incident data.",
  "Ticket Type is supplied, not independently verified ground truth.",
  "Excluding conflicting groups does not prove which original label is correct.",
  "Phone-like patterns are flagged, not redacted; residual PII may remain."
 ]
}
MANIFEST_PATH.parent.mkdir(parents=True,exist_ok=True)
MANIFEST_PATH.write_text(json.dumps(manifest,indent=2,ensure_ascii=False),encoding="utf-8")

saved=json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
assert saved["records"],"No lineage records."
for r in saved["records"]:
    src=BASE_DIR/r["source_relative_path"]
    dst=BASE_DIR/r["prepared_output"]
    assert src.is_file() and dst.is_file()
    assert sha256(src)==r["source_sha256"]
    assert sha256(dst)==r["prepared_sha256"]
print("COMMON LINEAGE VALIDATION: PASS")
print("Verified records:",len(saved["records"]))
print("Manifest:",MANIFEST_PATH)
print("Source SHA256:",record["source_sha256"])
print("Prepared SHA256:",record["prepared_sha256"])

COMMON LINEAGE VALIDATION: PASS
Verified records: 1
Manifest: /content/week3_project/data/week3_manifest.json
Source SHA256: b06a9cde84da65db388bd964d75f88ee1eed96607cf75d0c35f09c3f11bf8bea
Prepared SHA256: 45c80b20f6f492e7893576f55934d512bb102b1ae7b197b8ba593dc59289880e


## Storage and privacy checkpoint
Files under `/content` are temporary. Persist the prepared export and manifest only to a private, access-controlled location. Do not commit raw/prepared CSV files to public GitHub.

In [39]:
# Optional: persist outputs to a private Google Drive folder.
# Do not set folder sharing to "Anyone with the link".
from google.colab import drive
drive.mount("/content/drive")
DRIVE_OUT = Path("/content/drive/MyDrive/TicketIQ_Week3_Private")
DRIVE_OUT.mkdir(parents=True,exist_ok=True)
shutil.copy2(PREPARED_PATH,DRIVE_OUT/"prepared_table.csv")
shutil.copy2(MANIFEST_PATH,DRIVE_OUT/"week3_manifest.json")
print("Private Drive output:",DRIVE_OUT)
print("Keep folder access restricted; prepared text may still contain personal information.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Private Drive output: /content/drive/MyDrive/TicketIQ_Week3_Private
Keep folder access restricted; prepared text may still contain personal information.


In [40]:

# Cell 7 — REQUIRED: Common V1 Pipeline Utilities
# Run ONCE after completing Part B

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json


# Step 1: Define standard project paths
BASE_DIR = Path("/content/week3_project")
RAW_DIR = BASE_DIR / "data" / "raw"
PIPE_DIR = BASE_DIR / "data" / "pipeline_v1"

PIPE_DIR.mkdir(parents=True, exist_ok=True)

# Store registered pipeline runs for repeatability verification
PIPELINE_RUNS = {}


# Step 2: Calculate SHA-256 checksum
def sha256_file(path):
    return hashlib.sha256(
        Path(path).read_bytes()
    ).hexdigest()


# Step 3: Resolve an explicitly selected source
def resolve_source(relative_path, folder=False):
    source = RAW_DIR / relative_path

    if folder:
        if not source.is_dir():
            raise FileNotFoundError(
                f"Source folder not found: {source}"
            )
    else:
        if not source.is_file():
            raise FileNotFoundError(
                f"Source file not found: {source}"
            )

    return source


# Step 4: Execute the selected V1 data pipeline
def execute_pipeline(
    kind,
    source,
    inspect_fn,
    prepare_fn,
    configuration
):
    source = Path(source)

    # INGEST: Identify relevant source files
    if kind == "images":
        inputs = sorted(
            p for p in source.iterdir()
            if p.is_file()
            and p.suffix.lower() in {
                ".png", ".jpg", ".jpeg", ".webp"
            }
        )
    else:
        inputs = [source]

    if not inputs or any(not p.is_file() for p in inputs):
        raise ValueError(
            "No valid source files to ingest."
        )

    # INSPECT: Apply data-type-specific inspection
    inspection = inspect_fn(inputs, configuration)

    # Define output directory
    output_dir = PIPE_DIR / kind
    output_dir.mkdir(parents=True, exist_ok=True)

    # PREPARE / STORE: Apply selected transformation
    outputs, mappings, method = prepare_fn(
        inputs,
        output_dir,
        configuration
    )

    outputs = [Path(p) for p in outputs]

    # VALIDATE: Check prepared outputs
    if not outputs:
        raise ValueError(
            "No prepared outputs were produced."
        )

    for output in outputs:
        if not output.is_file():
            raise FileNotFoundError(
                f"Prepared output missing: {output}"
            )

        if output.stat().st_size == 0:
            raise ValueError(
                f"Prepared output is empty: {output}"
            )

    # Record basic validation evidence
    validation = {
        "output_count": len(outputs),
        "output_sha256": [
            sha256_file(p) for p in outputs
        ],
        "output_bytes": [
            p.stat().st_size for p in outputs
        ]
    }

    # RECORD LINEAGE: Create pipeline execution record
    record = {
        "pipeline_version": "week3-v1",
        "generated_at_utc": datetime.now(
            timezone.utc
        ).isoformat(),
        "data_type": kind,
        "configuration": configuration,
        "processing_method": method,
        "sources": [
            {
                "path": str(p),
                "sha256": sha256_file(p)
            }
            for p in inputs
        ],
        "outputs": [
            str(p) for p in outputs
        ],
        "source_output_mapping": mappings,
        "inspection": inspection,
        "validation": validation,
        "source_version": "Not verified",
        "access_rights": (
            "Not verified — check before sharing"
        )
    }

    # Save pipeline manifest
    manifest_path = output_dir / "pipeline_manifest.json"

    manifest_path.write_text(
        json.dumps(
            record,
            indent=2,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )

    print(
        f"SUCCESS: {kind} | "
        f"Inputs: {len(inputs)} | "
        f"Outputs: {len(outputs)}"
    )
    print("Manifest:", manifest_path)

    return record


# Step 5: Register and run a selected pathway
def register_and_run(
    kind,
    source,
    inspect_fn,
    prepare_fn,
    configuration
):
    # Keep the runner for repeatability verification
    def runner():
        return execute_pipeline(
            kind,
            source,
            inspect_fn,
            prepare_fn,
            configuration
        )

    first = runner()

    PIPELINE_RUNS[kind] = {
        "first": first,
        "runner": runner
    }

    return first


print("Cell 7 ready.")
print("Run ONLY the selected Cell 8 pathway(s).")
print("Common pipeline utilities are available.")


Cell 7 ready.
Run ONLY the selected Cell 8 pathway(s).
Common pipeline utilities are available.


In [41]:

# Cell 8B — CONDITIONAL REQUIRED: CSV / Excel Projects
# TicketIQ implementation based on Part B decisions

import pandas as pd
import re

# Step 1: Prepare source directory
RAW_DIR.mkdir(parents=True, exist_ok=True)

source_file = RAW_DIR / "customer_support_tickets.csv"

# Upload source CSV only if it is not already available
if not source_file.is_file():
    from google.colab import files

    print("Please upload customer_support_tickets.csv")
    uploaded = files.upload()

    if "customer_support_tickets.csv" not in uploaded:
        raise FileNotFoundError(
            "Expected customer_support_tickets.csv"
        )

    source_file.write_bytes(
        uploaded["customer_support_tickets.csv"]
    )

SOURCE_B = resolve_source("customer_support_tickets.csv")

# Step 2: Configure TicketIQ preparation
CONFIG_B = {
    "selected_columns": [
        "Ticket Subject",
        "Ticket Description",
        "Ticket Type"
    ],
    "trim_text": True,
    "redact_emails": True,
    "exclude_conflicting_groups": True,
    "remove_exact_duplicates": True
}

# Step 3: Define table loader
def read_table(src):
    if src.suffix.lower() == ".csv":
        return pd.read_csv(src)

    elif src.suffix.lower() == ".xlsx":
        return pd.read_excel(src)

    else:
        raise ValueError("Expected a CSV or XLSX source")


# Step 4: Inspect source dataset
def inspect_table(inputs, cfg):
    df = read_table(inputs[0])

    if df.empty:
        raise ValueError("Empty input table")

    missing_columns = [
        col for col in cfg["selected_columns"]
        if col not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Missing required columns: {missing_columns}"
        )

    return {
        "rows": len(df),
        "columns": df.columns.tolist(),
        "missing": {
            str(k): int(v)
            for k, v in df.isna().sum().items()
        },
        "exact_duplicates": int(df.duplicated().sum())
    }


# Step 5: Prepare the TicketIQ dataset
def prepare_table(inputs, out, cfg):
    src = inputs[0]
    df = read_table(src)

    # Select only the approved modelling columns
    result = df[cfg["selected_columns"]].copy()

    # Trim outer whitespace
    if cfg["trim_text"]:
        for col in [
            "Ticket Subject",
            "Ticket Description",
            "Ticket Type"
        ]:
            result[col] = result[col].astype("string").str.strip()

    # Redact detected email addresses
    email_pattern = r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b"

    if cfg["redact_emails"]:
        for col in ["Ticket Subject", "Ticket Description"]:
            result[col] = result[col].str.replace(
                email_pattern,
                "[EMAIL_REDACTED]",
                regex=True
            )

    # Create normalized keys for conflict detection
    if cfg["exclude_conflicting_groups"]:
        normalized_subject = (
            result["Ticket Subject"]
            .fillna("")
            .str.lower()
            .str.replace(r"\s+", " ", regex=True)
            .str.strip()
        )

        normalized_description = (
            result["Ticket Description"]
            .fillna("")
            .str.lower()
            .str.replace(r"\s+", " ", regex=True)
            .str.strip()
        )

        result["_normalized_subject"] = normalized_subject
        result["_normalized_description"] = normalized_description

        # Identify groups with more than one Ticket Type
        label_counts = result.groupby(
            ["_normalized_subject", "_normalized_description"],
            dropna=False
        )["Ticket Type"].nunique()

        conflicting_keys = label_counts[
            label_counts > 1
        ].index

        row_keys = pd.MultiIndex.from_arrays([
            result["_normalized_subject"],
            result["_normalized_description"]
        ])

        conflict_mask = row_keys.isin(conflicting_keys)

        conflict_groups = len(conflicting_keys)
        conflict_records = int(conflict_mask.sum())

        result = result.loc[~conflict_mask].copy()

        result.drop(
            columns=[
                "_normalized_subject",
                "_normalized_description"
            ],
            inplace=True
        )
    else:
        conflict_groups = 0
        conflict_records = 0

    # Remove exact duplicate modelling rows
    duplicates_removed = int(result.duplicated().sum())

    if cfg["remove_exact_duplicates"]:
        result = result.drop_duplicates().copy()

    if result.empty:
        raise ValueError("No prepared table rows")

    # Audit against Part B evidence
    audit = {
        "source_rows": len(df),
        "conflicting_groups_excluded": conflict_groups,
        "conflicting_records_excluded": conflict_records,
        "exact_duplicate_rows_removed": duplicates_removed,
        "prepared_rows": len(result),
        "prepared_columns": result.columns.tolist()
    }

    print("=== TICKETIQ PREPARATION AUDIT ===")
    print(audit)

    expected = {
        "source_rows": 8469,
        "conflicting_groups_excluded": 85,
        "conflicting_records_excluded": 205,
        "exact_duplicate_rows_removed": 24,
        "prepared_rows": 8240
    }

    for key, expected_value in expected.items():
        if audit[key] != expected_value:
            raise ValueError(
                f"Part B mismatch for {key}: "
                f"expected {expected_value}, got {audit[key]}. "
                "Output was not saved. Review the transformation logic."
            )

    # Save prepared output
    target = out / "prepared_table.csv"
    result.to_csv(target, index=False)

    # Describe actual processing
    method = (
        "selected_columns=Ticket Subject, Ticket Description, Ticket Type; "
        "trim_outer_whitespace=True; "
        "redact_detected_emails=True; "
        f"exclude_conflicting_groups={conflict_groups}; "
        f"remove_exact_duplicates={duplicates_removed}"
    )

    mapping = [{
        "source": str(src),
        "output": str(target)
    }]

    return [target], mapping, method


# Step 6: Execute and register selected pipeline
run_B = register_and_run(
    "tabular",
    SOURCE_B,
    inspect_table,
    prepare_table,
    CONFIG_B
)

print("Cell 8B completed successfully.")


=== TICKETIQ PREPARATION AUDIT ===
{'source_rows': 8469, 'conflicting_groups_excluded': 85, 'conflicting_records_excluded': 205, 'exact_duplicate_rows_removed': 24, 'prepared_rows': 8240, 'prepared_columns': ['Ticket Subject', 'Ticket Description', 'Ticket Type']}
SUCCESS: tabular | Inputs: 1 | Outputs: 1
Manifest: /content/week3_project/data/pipeline_v1/tabular/pipeline_manifest.json
Cell 8B completed successfully.


In [42]:

# Cell 8B — CONDITIONAL REQUIRED: CSV / Excel Projects
# TicketIQ implementation based on Part B decisions

import pandas as pd
import re

# Step 1: Prepare source directory
RAW_DIR.mkdir(parents=True, exist_ok=True)

source_file = RAW_DIR / "customer_support_tickets.csv"

# Upload source CSV only if it is not already available
if not source_file.is_file():
    from google.colab import files

    print("Please upload customer_support_tickets.csv")
    uploaded = files.upload()

    if "customer_support_tickets.csv" not in uploaded:
        raise FileNotFoundError(
            "Expected customer_support_tickets.csv"
        )

    source_file.write_bytes(
        uploaded["customer_support_tickets.csv"]
    )

SOURCE_B = resolve_source("customer_support_tickets.csv")

# Step 2: Configure TicketIQ preparation
CONFIG_B = {
    "selected_columns": [
        "Ticket Subject",
        "Ticket Description",
        "Ticket Type"
    ],
    "trim_text": True,
    "redact_emails": True,
    "exclude_conflicting_groups": True,
    "remove_exact_duplicates": True
}

# Step 3: Define table loader
def read_table(src):
    if src.suffix.lower() == ".csv":
        return pd.read_csv(src)

    elif src.suffix.lower() == ".xlsx":
        return pd.read_excel(src)

    else:
        raise ValueError("Expected a CSV or XLSX source")


# Step 4: Inspect source dataset
def inspect_table(inputs, cfg):
    df = read_table(inputs[0])

    if df.empty:
        raise ValueError("Empty input table")

    missing_columns = [
        col for col in cfg["selected_columns"]
        if col not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Missing required columns: {missing_columns}"
        )

    return {
        "rows": len(df),
        "columns": df.columns.tolist(),
        "missing": {
            str(k): int(v)
            for k, v in df.isna().sum().items()
        },
        "exact_duplicates": int(df.duplicated().sum())
    }


# Step 5: Prepare the TicketIQ dataset
def prepare_table(inputs, out, cfg):
    src = inputs[0]
    df = read_table(src)

    # Select only the approved modelling columns
    result = df[cfg["selected_columns"]].copy()

    # Trim outer whitespace
    if cfg["trim_text"]:
        for col in [
            "Ticket Subject",
            "Ticket Description",
            "Ticket Type"
        ]:
            result[col] = result[col].astype("string").str.strip()

    # Redact detected email addresses
    email_pattern = r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b"

    if cfg["redact_emails"]:
        for col in ["Ticket Subject", "Ticket Description"]:
            result[col] = result[col].str.replace(
                email_pattern,
                "[EMAIL_REDACTED]",
                regex=True
            )

    # Create normalized keys for conflict detection
    if cfg["exclude_conflicting_groups"]:
        normalized_subject = (
            result["Ticket Subject"]
            .fillna("")
            .str.lower()
            .str.replace(r"\s+", " ", regex=True)
            .str.strip()
        )

        normalized_description = (
            result["Ticket Description"]
            .fillna("")
            .str.lower()
            .str.replace(r"\s+", " ", regex=True)
            .str.strip()
        )

        result["_normalized_subject"] = normalized_subject
        result["_normalized_description"] = normalized_description

        # Identify groups with more than one Ticket Type
        label_counts = result.groupby(
            ["_normalized_subject", "_normalized_description"],
            dropna=False
        )["Ticket Type"].nunique()

        conflicting_keys = label_counts[
            label_counts > 1
        ].index

        row_keys = pd.MultiIndex.from_arrays([
            result["_normalized_subject"],
            result["_normalized_description"]
        ])

        conflict_mask = row_keys.isin(conflicting_keys)

        conflict_groups = len(conflicting_keys)
        conflict_records = int(conflict_mask.sum())

        result = result.loc[~conflict_mask].copy()

        result.drop(
            columns=[
                "_normalized_subject",
                "_normalized_description"
            ],
            inplace=True
        )
    else:
        conflict_groups = 0
        conflict_records = 0

    # Remove exact duplicate modelling rows
    duplicates_removed = int(result.duplicated().sum())

    if cfg["remove_exact_duplicates"]:
        result = result.drop_duplicates().copy()

    if result.empty:
        raise ValueError("No prepared table rows")

    # Audit against Part B evidence
    audit = {
        "source_rows": len(df),
        "conflicting_groups_excluded": conflict_groups,
        "conflicting_records_excluded": conflict_records,
        "exact_duplicate_rows_removed": duplicates_removed,
        "prepared_rows": len(result),
        "prepared_columns": result.columns.tolist()
    }

    print("=== TICKETIQ PREPARATION AUDIT ===")
    print(audit)

    expected = {
        "source_rows": 8469,
        "conflicting_groups_excluded": 85,
        "conflicting_records_excluded": 205,
        "exact_duplicate_rows_removed": 24,
        "prepared_rows": 8240
    }

    for key, expected_value in expected.items():
        if audit[key] != expected_value:
            raise ValueError(
                f"Part B mismatch for {key}: "
                f"expected {expected_value}, got {audit[key]}. "
                "Output was not saved. Review the transformation logic."
            )

    # Save prepared output
    target = out / "prepared_table.csv"
    result.to_csv(target, index=False)

    # Describe actual processing
    method = (
        "selected_columns=Ticket Subject, Ticket Description, Ticket Type; "
        "trim_outer_whitespace=True; "
        "redact_detected_emails=True; "
        f"exclude_conflicting_groups={conflict_groups}; "
        f"remove_exact_duplicates={duplicates_removed}"
    )

    mapping = [{
        "source": str(src),
        "output": str(target)
    }]

    return [target], mapping, method


# Step 6: Execute and register selected pipeline
run_B = register_and_run(
    "tabular",
    SOURCE_B,
    inspect_table,
    prepare_table,
    CONFIG_B
)

print("Cell 8B completed successfully.")


=== TICKETIQ PREPARATION AUDIT ===
{'source_rows': 8469, 'conflicting_groups_excluded': 85, 'conflicting_records_excluded': 205, 'exact_duplicate_rows_removed': 24, 'prepared_rows': 8240, 'prepared_columns': ['Ticket Subject', 'Ticket Description', 'Ticket Type']}
SUCCESS: tabular | Inputs: 1 | Outputs: 1
Manifest: /content/week3_project/data/pipeline_v1/tabular/pipeline_manifest.json
Cell 8B completed successfully.


In [43]:
# Cell 9 — REQUIRED: Repeatability and Provenance Verification

# Step 1: Ensure at least one pipeline has been registered
assert PIPELINE_RUNS, (
    "No pipeline runs registered. "
    "Run at least one selected Cell 8 pathway first."
)

print("--- Repeatability Verification ---")

# Step 2: Rerun each selected pipeline
for kind, entry in PIPELINE_RUNS.items():
    first = entry["first"]

    # Execute the same pipeline again
    second = entry["runner"]()

    # Step 3: Verify input provenance
    assert first["sources"] == second["sources"], (
        f"Source fingerprint changed for {kind}"
    )

    # Step 4: Verify prepared output fingerprints
    assert first["validation"] == second["validation"], (
        f"Prepared output changed for {kind}"
    )

    # Step 5: Verify source-to-output mapping
    assert (
        first["source_output_mapping"]
        == second["source_output_mapping"]
    ), (
        f"Source-to-output mapping changed for {kind}"
    )

    print(
        f"PASS: {kind} — source fingerprints, "
        "prepared outputs and mappings match"
    )

# Step 6: Display final verification message
print("\nAll selected pipelines passed repeatability smoke checks.")

--- Repeatability Verification ---
=== TICKETIQ PREPARATION AUDIT ===
{'source_rows': 8469, 'conflicting_groups_excluded': 85, 'conflicting_records_excluded': 205, 'exact_duplicate_rows_removed': 24, 'prepared_rows': 8240, 'prepared_columns': ['Ticket Subject', 'Ticket Description', 'Ticket Type']}
SUCCESS: tabular | Inputs: 1 | Outputs: 1
Manifest: /content/week3_project/data/pipeline_v1/tabular/pipeline_manifest.json
PASS: tabular — source fingerprints, prepared outputs and mappings match

All selected pipelines passed repeatability smoke checks.


In [44]:

from pathlib import Path
import shutil
from google.colab import drive

# Mount Google Drive
drive.mount("/content/drive")

# Part C output location
SOURCE = Path("/content/week3_project/data/pipeline_v1/tabular")

# Private Google Drive destination
DEST = Path(
    "/content/drive/MyDrive/TicketIQ_Week3_Private/"
    "pipeline_v1/tabular"
)
DEST.mkdir(parents=True, exist_ok=True)

# Copy both Part C outputs
for filename in ["prepared_table.csv", "pipeline_manifest.json"]:
    source_file = SOURCE / filename

    if not source_file.exists():
        raise FileNotFoundError(source_file)

    shutil.copy2(source_file, DEST / filename)
    print("Copied:", filename)

print("\nSaved to:", DEST)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Copied: prepared_table.csv
Copied: pipeline_manifest.json

Saved to: /content/drive/MyDrive/TicketIQ_Week3_Private/pipeline_v1/tabular
